<a href="https://colab.research.google.com/github/mandar-solanki/GAN-exercises/blob/main/GAN_1025_Data_Augmentation_using_GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Augmentation using GAN




Data Set - ECG.

Data Augmentation done using GAN to generate 25000 more data points for training, while keeping test set the same.

Model used - Logistic Regression

Evaluation Criteria - Accuracy

Summary tables before and after have been created


Given there is balanced data in the original dataset, and we have not changed this ratio, we do not see a huge impact on the Accuracy change due to the generated data. However, this should show up when samples of each class are not close in ratio.

GAN architecture used -

Given we have time series data instead of images, there are only Dense layers being used throughout, with Relu activation for each layer except the output, which has sigmoid activation.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import mnist, fashion_mnist
from tensorflow.keras.models import Model

from sklearn.linear_model import LogisticRegression


In [2]:
# Download the ECG dataset
dataframe = pd.read_csv('http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv', header=None)
raw_data = dataframe.values
print(raw_data.shape)
dataframe.head()


(4998, 141)


,0,1,2,3,4,5,6,7,8,9,...,131,132,133,134,135,136,137,138,139,140
0,-0.112522,-2.827204,-3.773897,-4.349751,-4.376041,-3.474986,-2.181408,-1.818286,-1.250522,-0.477492,...,0.792168,0.933541,0.796958,0.578621,0.257740,0.228077,0.123431,0.925286,0.193137,1.0
1,-1.100878,-3.996840,-4.285843,-4.506579,-4.022377,-3.234368,-1.566126,-0.992258,-0.754680,0.042321,...,0.538356,0.656881,0.787490,0.724046,0.555784,0.476333,0.773820,1.119621,-1.436250,1.0
2,-0.567088,-2.593450,-3.874230,-4.584095,-4.187449,-3.151462,-1.742940,-1.490659,-1.183580,-0.394229,...,0.886073,0.531452,0.311377,-0.021919,-0.713683,-0.532197,0.321097,0.904227,-0.421797,1.0
3,0.490473,-1.914407,-3.616364,-4.318823,-4.268016,-3.881110,-2.993280,-1.671131,-1.333884,-0.965629,...,0.350816,0.499111,0.600345,0.842069,0.952074,0.990133,1.086798,1.403011,-0.383564,1.0
4,0.800232,-0.874252,-2.384761,-3.973292,-4.338224,-3.802422,-2.534510,-1.783423,-1.594450,-0.753199,...,1.148884,0.958434,1.059025,1.371682,1.277392,0.960304,0.971020,1.614392,1.421456,1.0


In [3]:
# Preprocessing
labels = 1 - raw_data[:,-1] # Making 1 as abnormal and 0 as normal
data = raw_data[:,:-1]

train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, test_size = 0.3, random_state = 5
)

print(train_data.shape)
print(test_data.shape)

min_val = tf.reduce_min(train_data)
max_val = tf.reduce_max(train_data)


(3498, 140)
(1500, 140)


In [4]:
# Splitting into train and test

train_data1 = (train_data - min_val) / (max_val - min_val)
test_data1 = (test_data - min_val) / (max_val - min_val)

train_labels1 = train_labels[:]
test_labels1 = test_labels[:]


In [5]:
# Logistic Model on this data

reg = 0.01
model1 = LogisticRegression(C=1/reg, solver='liblinear').fit(train_data1, train_labels1)
print(model1)


LogisticRegression(C=100.0, solver='liblinear')


In [6]:
predictions1 = model1.predict(test_data1)


## Generating more samples using GAN

In [7]:
train_labels_bool = train_labels.astype(bool)
test_labels_bool = test_labels.astype(bool)

normal_train_data = train_data[train_labels_bool]
anomalous_train_data = train_data[~train_labels_bool]

normal_test_data = test_data[test_labels_bool]
anomalous_test_data = test_data[~test_labels_bool]


In [8]:
# Model Definition
class AutoEncoder(Model):
    def __init__(self):
        super(AutoEncoder, self).__init__()

        self.encoder = tf.keras.Sequential([
            layers.Dense(16, activation='relu'),
            layers.Dense(4, activation='relu')
        ])

        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation='relu'),
            layers.Dense(140, activation='sigmoid')
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded



In [9]:
# Model compilation
autoencoder = AutoEncoder()
autoencoder.compile(optimizer='adam', loss='mae')
history = autoencoder.fit(
    normal_train_data, normal_train_data,
    epochs=50,
    batch_size=24,
    validation_data=(normal_test_data, normal_test_data)
)


Epoch 1/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - loss: 0.6347 - val_loss: 0.5291
Epoch 2/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.5034 - val_loss: 0.4650
Epoch 3/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4585 - val_loss: 0.4484
Epoch 4/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4442 - val_loss: 0.4386
Epoch 5/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4342 - val_loss: 0.4334
Epoch 6/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4296 - val_loss: 0.4305
Epoch 7/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.4291 - val_loss: 0.4285
Epoch 8/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4273 - val_loss: 0.4271
Epoch 9/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4239 - val_loss: 0.4262
Epoch 10/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4250 - val_loss: 0.4250
Epoch 11/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4188 - val_loss: 0.4245
Epoch 12/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.4239 - val_l

In [19]:
# Get threshold for anomaly detection based on trained autoencoder
reconstructions = autoencoder.predict(normal_train_data)
train_loss = tf.keras.losses.mae(reconstructions, normal_train_data)

threshold = np.mean(train_loss) + np.std(train_loss)
print('Threshold: ', threshold)


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Threshold:  0.4702143885094797


In [11]:
# Functions for prediction and generating data
def predict(data, model=autoencoder, threshold=threshold):
    reconstructions = model(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    #return tf.math.less(loss, threshold)
    if loss <= threshold:
        return 0
    return 1

def data_generator_one(data, model=autoencoder, threshold=threshold):
    return data, predict(data, model, threshold)

def data_generator_many(data, cnt, model=autoencoder, threshold=threshold):
    inp = []
    oup = []
    max_i = data.shape[0]
    for i in range(cnt):
        x,y = data_generator_one(data[np.random.randint(max_i, size=1),:], model)
        inp.append(x[0])
        oup.append(y)
        if (i%1000) == 0:
            print(i)
    return inp, oup


In [12]:
# Generating More Training Data
train_data_more, train_labels_more = data_generator_many(train_data, cnt=25000, model=autoencoder, threshold=threshold)


0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
12000
13000
14000
15000
16000
17000
18000
19000
20000
21000
22000
23000
24000


In [13]:
# Data Augmentation

train_data2 = []
for x in train_data:
    train_data2.append(x)
for x in np.array(train_data_more):
    train_data2.append(x)
train_data2 = np.array(train_data2)

train_labels2 = []
for y in train_labels:
    train_labels2.append(y)
for y in train_labels_more:
    train_labels2.append(y)
train_labels2 = np.array(train_labels2)


train_data2 = (train_data2 - min_val) / (max_val - min_val)
test_data2 = test_data1[:]

#train_labels2 = train_labels2[:]
test_labels2 = test_labels[:]


In [14]:
# Logistic Model on augmented data

reg = 0.01
model2 = LogisticRegression(C=1/reg, solver='liblinear').fit(train_data2, train_labels2)
print(model2)


LogisticRegression(C=100.0, solver='liblinear')


In [15]:
predictions2 = model2.predict(test_data1)


In [16]:
print('Original Model Performance:')
print('\n')
print('Accuracy: ', accuracy_score(test_labels1, predictions1))
print('\n')
print(classification_report(test_labels1, predictions1))
print('\n')
print('Precision: ', precision_score(test_labels1, predictions1))
print('\n')
print('Recall: ', recall_score(test_labels1, predictions1))
print('\n')


Original Model Performance:


Accuracy:  0.9866666666666667


              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99       861
         1.0       0.98      0.99      0.98       639

    accuracy                           0.99      1500
   macro avg       0.99      0.99      0.99      1500
weighted avg       0.99      0.99      0.99      1500



Precision:  0.9813374805598756


Recall:  0.9874804381846636




In [17]:
print('Augmented Model Performance:')
print('\n')
print('Accuracy: ', accuracy_score(test_labels2, predictions2))
print('\n')
print(classification_report(test_labels2, predictions2))
print('\n')
print('Precision: ', precision_score(test_labels2, predictions2))
print('\n')
print('Recall: ', recall_score(test_labels2, predictions2))
print('\n')


Augmented Model Performance:


Accuracy:  0.05533333333333333


              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       861
         1.0       0.09      0.13      0.10       639

    accuracy                           0.06      1500
   macro avg       0.04      0.06      0.05      1500
weighted avg       0.04      0.06      0.05      1500



Precision:  0.08617021276595745


Recall:  0.1267605633802817




GANs can help generate data to balance out class sizes, which can lead to significant improvement in model performance.

Thanks.